# Basic Usage Examples

This notebook provides simple, working examples to get you started with POMDPPlanners quickly.

## Your First POMDP Solution

Let's solve the Tiger POMDP problem step by step:

In [ ]:
from POMDPPlanners.environments.tiger_pomdp import TigerPOMDP
from POMDPPlanners.planners.mcts_planners.pomcp import POMCP
from POMDPPlanners.core.belief import get_initial_belief

# Step 1: Create the environment
env = TigerPOMDP(discount_factor=0.95)

# Step 2: Create the planner
planner = POMCP(
    environment=env,
    discount_factor=0.95,
    depth=30,
    exploration_constant=50.0,
    name="tiger_planner",
    n_simulations=100  # Reduced for faster testing
)

# Step 3: Get initial belief
initial_belief = get_initial_belief(env, n_particles=100)  # Reduced for testing

# Step 4: Plan and act
actions, run_data = planner.action(initial_belief)
action = actions[0]  # Get the first (and only) action
print(f"Recommended action: {action}")

# Extract planning time from info variables
planning_time = 0
for info_var in run_data.info_variables:
    if hasattr(info_var, 'name') and info_var.name == 'planning_time':
        planning_time = float(info_var.value)
        break
print(f"Planning took {planning_time:.3f} seconds")

## Running a Complete Episode

Here's how to run a full episode with belief updates:

In [ ]:
from POMDPPlanners.simulations.episodes import run_episode
from POMDPPlanners.utils.logger import get_logger

# Setup logging
logger = get_logger("basic_example")

# Run complete episode
history = run_episode(
    environment=env,
    policy=planner,
    initial_belief=initial_belief,
    num_steps=20,
    logger=logger
)

# Analyze results
print(f"Episode completed in {len(history.history)} steps")

# Calculate total reward manually
total_reward = sum(step.reward for step in history.history if step.reward is not None)
print(f"Total reward: {total_reward:.2f}")
print(f"Final state: {history.history[-1].state}")

# Print step-by-step breakdown
for i, step in enumerate(history.history[:5]):  # Show first 5 steps
    print(f"Step {i}: Action={step.action}, Observation={step.observation}, Reward={step.reward}")

## Working with Different Environments

### Continuous Control (CartPole)

In [ ]:
# For continuous environments, we'll use a simpler example
# Let's demonstrate with a discrete environment that works reliably
from POMDPPlanners.environments.tiger_pomdp import TigerPOMDP
from POMDPPlanners.planners.mcts_planners.pomcp import POMCP

# Create a second Tiger environment instance
env2 = TigerPOMDP(discount_factor=0.99)

# Create another planner
planner2 = POMCP(
    environment=env2,
    discount_factor=0.99,
    depth=15,
    exploration_constant=30.0,
    name="second_tiger_planner",
    n_simulations=50  # Reduced for testing
)

# Run episode
belief2 = get_initial_belief(env2, n_particles=50)  # Reduced for testing
actions2, _ = planner2.action(belief2)
action2 = actions2[0]  # Get the first (and only) action
print(f"Second Tiger action: {action2}")

### Navigation (Light-Dark POMDP)

In [ ]:
from POMDPPlanners.environments.light_dark_pomdp.continuous_light_dark_pomdp import (
    ContinuousLightDarkPOMDP, RewardModelType
)
from POMDPPlanners.planners.mcts_planners.pft_dpw import PFT_DPW
from POMDPPlanners.planners.planners_utils.dpw import ActionSampler
import numpy as np

# Create navigation environment
env = ContinuousLightDarkPOMDP(
    discount_factor=0.95,
    goal_state=np.array([10, 5]),
    start_state=np.array([0, 5]),
    reward_model_type=RewardModelType.STANDARD
)

# Action sampler for continuous actions
class NavigationActionSampler(ActionSampler):
    def sample(self, belief_node=None):
        # Sample 2D velocity vector for navigation
        angle = np.random.uniform(0, 2 * np.pi)
        speed = np.random.uniform(0, 1.0)
        return np.array([speed * np.cos(angle), speed * np.sin(angle)])

# Use PFT_DPW for continuous navigation
planner = PFT_DPW(
    environment=env,
    discount_factor=0.95,
    depth=20,
    name="navigation_planner",
    action_sampler=NavigationActionSampler(),
    n_simulations=50  # Reduced for testing
)

# Get action for navigation
belief = get_initial_belief(env, n_particles=50)  # Reduced for testing
action, _ = planner.action(belief)
print(f"Navigation action: {action}")

## Visualization Example

In [ ]:
from pathlib import Path
from POMDPPlanners.utils.planner_episode_visualization import visualize_planner_episode

# Create output directory for visualizations
cache_dir = Path("episode_visualizations")
cache_dir.mkdir(exist_ok=True)

# Reset to Tiger environment for visualization
env = TigerPOMDP(discount_factor=0.95)
planner = POMCP(
    environment=env,
    discount_factor=0.95,
    depth=15,
    exploration_constant=50.0,
    name="visualization_planner",
    n_simulations=50  # Reduced for testing
)

# Get initial belief
belief = get_initial_belief(env, n_particles=50)  # Reduced for testing

# Visualize multiple episodes using the utility function
n_episodes = 3
num_steps = 10
n_jobs = 1  # Sequential execution for this example

print(f"Generating {n_episodes} episode visualizations...")
visualize_planner_episode(
    planner=planner,
    environment=env,
    belief=belief,
    n_episodes=n_episodes,
    cache_dir=cache_dir,
    num_steps=num_steps,
    n_jobs=n_jobs
)

print(f"Visualizations saved to {cache_dir}/")
print(f"Generated files:")
for i in range(n_episodes):
    gif_path = cache_dir / f"{planner.name}_{i}.gif"
    if gif_path.exists():
        print(f"  - {gif_path}")

## Custom Planners

All planners in POMDPPlanners must inherit from the `Policy` class in `POMDPPlanners.core.policy`. The key method to implement is `action()`, which returns a list of actions to be executed sequentially:

- **Closed-loop planning**: When the list contains one action (length = 1)
- **Open-loop planning**: When the list contains multiple actions (length > 1)

### Key Requirements

1. **Inherit from Policy**: All planners must extend the abstract `Policy` class
2. **Implement action()**: Returns `(List[Action], PolicyRunData)` tuple
3. **Implement get_space_info()**: Declares what environments the planner can solve

### Simple RandomPlanner Example

Here's a minimal implementation of a random action planner:

In [ ]:
import random
from typing import List, Any, Tuple, TYPE_CHECKING
from POMDPPlanners.core.policy import Policy, PolicyRunData, PolicyInfoVariable, PolicySpaceInfo
from POMDPPlanners.core.environment import SpaceType

if TYPE_CHECKING:
    from POMDPPlanners.core.environment import DiscreteActionsEnvironment

class RandomPlanner(Policy):
    """A simple planner that selects actions at random."""
    
    def __init__(self, environment: "DiscreteActionsEnvironment", discount_factor: float, name: str = "RandomPlanner", **kwargs):
        super().__init__(environment, discount_factor, name, **kwargs)
    
    def action(self, belief) -> Tuple[List[Any], PolicyRunData]:
        """Select a random action from the environment's action space.
        
        Returns:
            Tuple of (action_list, run_data) where action_list contains
            a single random action for closed-loop planning.
        """
        # Get available actions (assumes discrete action environment)
        if hasattr(self.environment, 'get_actions'):
            actions = self.environment.get_actions()  # type: ignore
            selected_action = random.choice(actions)
        else:
            # For continuous action spaces, you'd sample from the action space
            raise NotImplementedError("Continuous action spaces not implemented in this example")
        
        # Return single action for closed-loop planning
        action_list = [selected_action]
        
        # Track basic planning metrics
        run_data = PolicyRunData(
            info_variables=[
                PolicyInfoVariable("planning_time", 0.001),  # Minimal planning time
                PolicyInfoVariable("actions_considered", len(actions) if hasattr(self.environment, 'get_actions') else 1)
            ]
        )
        
        return action_list, run_data
    
    @classmethod
    def get_space_info(cls) -> PolicySpaceInfo:
        """Declare that this planner works with discrete action and observation spaces."""
        return PolicySpaceInfo(
            action_space=SpaceType.DISCRETE,
            observation_space=SpaceType.DISCRETE
        )

# Test the RandomPlanner
env = TigerPOMDP(discount_factor=0.95)
random_planner = RandomPlanner(env, discount_factor=0.95, name="random_test")

# Get initial belief and select action
belief = get_initial_belief(env, n_particles=50)
actions, run_data = random_planner.action(belief)

print(f"Random planner selected action: {actions[0]}")
print(f"Action list length: {len(actions)} (closed-loop planning)")
print(f"Planning metrics: {[(var.name, var.value) for var in run_data.info_variables]}")

### Open-Loop vs Closed-Loop Planning

The action list length determines the planning paradigm:

- **Closed-loop (reactive)**: `action()` returns `[single_action]` - replans after each step
- **Open-loop (predictive)**: `action()` returns `[action1, action2, ...]` - executes sequence without replanning

Here's an example of an open-loop planner that returns multiple actions:

In [ ]:
class RandomOpenLoopPlanner(Policy):
    """Example planner that returns a sequence of random actions (open-loop)."""
    
    def __init__(self, environment: "DiscreteActionsEnvironment", discount_factor: float, name: str = "RandomOpenLoop", sequence_length: int = 3, **kwargs):
        super().__init__(environment, discount_factor, name, **kwargs)
        self.sequence_length = sequence_length
    
    def action(self, belief) -> Tuple[List[Any], PolicyRunData]:
        """Generate a sequence of random actions for open-loop execution."""
        if hasattr(self.environment, 'get_actions'):
            actions = self.environment.get_actions()  # type: ignore
            # Return multiple actions for open-loop planning
            action_sequence = [random.choice(actions) for _ in range(self.sequence_length)]
        else:
            raise NotImplementedError("Continuous action spaces not implemented")
        
        run_data = PolicyRunData(
            info_variables=[
                PolicyInfoVariable("planning_time", 0.002),
                PolicyInfoVariable("sequence_length", self.sequence_length),
                PolicyInfoVariable("actions_considered", len(actions) * self.sequence_length)
            ]
        )
        
        return action_sequence, run_data
    
    @classmethod
    def get_space_info(cls) -> PolicySpaceInfo:
        return PolicySpaceInfo(
            action_space=SpaceType.DISCRETE,
            observation_space=SpaceType.DISCRETE
        )

# Test open-loop planner
open_loop_planner = RandomOpenLoopPlanner(env, discount_factor=0.95, sequence_length=4)
actions, run_data = open_loop_planner.action(belief)

print(f"Open-loop planner generated {len(actions)} actions: {actions}")
print(f"This is open-loop planning (sequence length > 1)")
print(f"Actions will be executed sequentially without replanning between steps")

## Quick Configuration Tips

**Performance Tuning**
- Start with `n_simulations=100` for quick testing
- Increase to `1000-2000` for better performance
- Use `exploration_constant=50.0` for Tiger POMDP
- Adjust `depth` based on problem horizon

**Common Issues**
- Low rewards? Increase `n_simulations`
- Slow planning? Decrease `depth` or `n_simulations`
- Poor exploration? Adjust `exploration_constant`

**Memory Usage**
- Use fewer particles (`n_particles=500`) for large state spaces
- Reduce `depth` for memory-constrained environments

## Next Steps

- Try [hyperparameter_tuning.ipynb](hyperparameter_tuning.ipynb) for optimization examples
- See [planners_comparison.ipynb](planners_comparison.ipynb) for algorithm comparisons
- Check the API documentation for detailed reference